# 02 — Customer Segmentation (Personas)
### SmartTel Retention Intelligence — Module 2

**Input:** `telco_features_latest.parquet` (produced by `01_eda_and_data_quality.ipynb`)

**Scope of this notebook** (Module 2 of the proposal):
1. Load the feature-store dataset
2. Select and prepare features for clustering (mixed categorical/numeric)
3. Fit K-Prototypes across a range of *k*, choose *k* using an elbow (cost) curve and an approximate silhouette check
4. Fit the final model and profile each cluster
5. Assign interpretable, data-driven persona names
6. Compute churn rate **within each persona** — the key insight this module must produce
7. Persist the persona-labeled dataset back to the feature store

> **Why K-Prototypes, not plain KMeans:** most fields in this dataset are categorical (`Contract`, `InternetService`, `PaymentMethod`, etc.), and KMeans' Euclidean distance isn't meaningful on unordered categories. K-Prototypes combines Euclidean distance for numeric fields with a matching-dissimilarity measure for categorical fields, which fits this dataset's structure directly.


Import abd setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import date

from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

FEATURE_STORE_DIR = Path("../data/feature_store")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load Feature-Store Dataset

In [2]:
df = pd.read_parquet(FEATURE_STORE_DIR / "telco_features_latest.parquet")
#print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,services_count,avg_monthly_revenue,contract_risk_flag
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-6mo,1,29.850000,1
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48mo,3,55.573529,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-6mo,3,54.075000,0
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48mo,3,40.905556,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-6mo,1,75.825000,1


This is the cleaned, feature-engineered output of Module 1: `TotalCharges` blanks imputed, `SeniorCitizen` normalized to Yes/No, and `tenure_bucket`, `services_count`, `avg_monthly_revenue`, and `contract_risk_flag` already added.


## 3. Feature Selection for Clustering

** Features excluded from clustering, and why:**
- `customerID` — identifier, not a feature
- `Churn` — the target. Clustering on the label would make "churn rate by persona" circular; personas must be discoverable independently of the outcome we later measure against them
- `TotalCharges` — confirmed in Module 1 to correlate >0.99 with `tenure × MonthlyCharges`; including it would let billing history dominate the distance metric with information already captured by `tenure` and `MonthlyCharges`
- `avg_monthly_revenue` — derived from `TotalCharges`/`tenure`; kept for later profiling but dropped from the clustering distance to avoid double-weighting tenure and spend
- `tenure_bucket` — a discretized version of `tenure`; keeping both would double-count the same signal in a mixed-distance metric. The continuous `tenure` is used for clustering; `tenure_bucket` is kept for readable profiling afterward
- `contract_risk_flag` — derived directly from `Contract` and `PaymentMethod`, both of which are already included; would triple-count that signal

**Numeric features used:** `tenure`, `MonthlyCharges`, `services_count`

**Categorical features used:** all remaining service, account, and demographic fields (16 columns)


In [4]:
numeric_features = ["tenure", "MonthlyCharges", "services_count"]

categorical_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
]
cluster_cols = numeric_features + categorical_features
print(len(cluster_cols))

19


#### checking for missing values

In [5]:
X = df[cluster_cols].copy()
X.isnull().sum().sum()  # sanity check: should be 0, already cleaned in Module 1

np.int64(0)

## 4. Data Preprocessing

Numeric features are standardized (zero mean, unit variance) so that `tenure` (range ~0–72) and `MonthlyCharges` (range ~18–120) contribute comparably to the Euclidean portion of the K-Prototypes distance, rather than the larger-magnitude feature dominating by scale alone.

K-Prototypes requires numeric columns first, followed by categorical columns, with the categorical column *indices* passed explicitly, the cell below prepares that layout.


In [6]:
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features] = scaler.fit_transform(X[numeric_features])

# K-Prototypes needs categorical columns as strings and their positional indices
for col in categorical_features:
    X_scaled[col] = X_scaled[col].astype(str)

categorical_indices = [X_scaled.columns.get_loc(c) for c in categorical_features] # get categorical column positions
X_matrix = X_scaled.to_numpy() #convert to matrix for k-prototypes

print(f"Matrix shape: {X_matrix.shape}")
print(f"Categorical column indices: {categorical_indices}")


Matrix shape: (7043, 19)
Categorical column indices: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
